# GPT Actions - Snowflake 直连
## 简介

本页面为构建特定应用 GPT Action 的开发者提供说明和指南。在继续之前，请先熟悉以下信息：



* [GPT Actions 简介](https://platform.openai.com/docs/actions)
* [GPT Actions Library 简介](https://platform.openai.com/docs/actions/actions-library)
* [从零构建 GPT Action 的示例](https://platform.openai.com/docs/actions/getting-started)

这个 GPT Action 概述了如何连接到 Snowflake 数据仓库。该 Action 接收用户的问题，扫描相关表以收集数据 schema，然后编写 SQL 查询来回答用户的问题。

注意：本 cookbook 返回的是 [ResultSet SQL statement](https://docs.snowflake.com/en/developer-guide/sql-api/handling-responses#getting-the-data-from-the-results)，而不是完整结果，因为完整结果可能超出 GPT Actions 的 `application/json` 负载限制。对于生产和高级用例，请使用中间件返回 CSV 文件。可按照 [GPT Actions - Snowflake Middleware cookbook](gpt_action_snowflake_middleware.ipynb) 实现该流程。

### 价值与业务用例示例


价值：用户现在可以利用 ChatGPT 的自然语言能力直接连接到 Snowflake 数据仓库。

用例示例：



* 数据科学家可以连接到表，并使用 ChatGPT 的数据分析功能进行数据分析
* 普通数据用户可以就其交易数据提出基本问题
* 用户能更清楚地了解自己的数据及潜在异常

## 应用信息

### 应用关键链接


开始之前，请先查看该应用的以下链接：

* 应用网站：[https://app.snowflake.com/](https://app.snowflake.com/)
* 应用 API 文档：[https://docs.snowflake.com/en/developer-guide/sql-api/intro](https://docs.snowflake.com/en/developer-guide/sql-api/intro)

### 应用前提条件

开始之前，请确保在你的应用环境中完成以下步骤：

* 开通一个 Snowflake 数据仓库
* 确保通过 ChatGPT 登录 Snowflake 的用户拥有具备必要权限的角色，可以访问相应的数据库、schema 和表

## 1. 配置自定义 GPT


### 设置 GPT 指令

创建自定义 GPT 后，将下面的文本复制到 Instructions 面板中。有疑问？请查看[入门示例](https://platform.openai.com/docs/actions/getting-started)，更详细地了解这一步是如何进行的。

In [ ]:
**Context**: You are an expert at writing Snowflake SQL queries. A user is going to ask you a question. 

**Instructions**:
1. No matter the user's question, start by running `runQuery` operation using this query: "SELECT column_name, table_name, data_type, comment FROM {database}.INFORMATION_SCHEMA.COLUMNS" 
-- Assume warehouse = "<insert your default warehouse here>", database = "<insert your default database here>", unless the user provides different values 
2. Convert the user's question into a SQL statement that leverages the step above and run the `runQuery` operation on that SQL statement to confirm the query works. Add a limit of 100 rows
3. Now remove the limit of 100 rows and return back the query for the user to see
4. Use the <your_role> role when querying Snowflake
5. Run each step in sequence. Explain what you are doing in a few sentences, run the action, and then explain what you learned. This will help the user understand the reason behind your workflow. 

**Additional Notes**: If the user says "Let's get started", explain that the user can provide a project or dataset, along with a question they want answered. If the user has no ideas, suggest that we have a sample flights dataset they can query - ask if they want you to query that


### OpenAPI Schema

创建自定义 GPT 后，将下面的文本复制到 Actions 面板中。按照[此处](https://docs.snowflake.com/en/user-guide/organizations-connect#standard-account-urls)的说明，将 servers url 更新为你的 Snowflake 账户名 URL 加上 `/api/v2`。有疑问？请查看[入门示例](https://platform.openai.com/docs/actions/getting-started)，更详细地了解这一步是如何进行的。

In [ ]:
openapi: 3.1.0
info:
  title: Snowflake Statements API
  version: 1.0.0
  description: API for executing statements in Snowflake with specific warehouse and role settings.
servers:
  - url: 'https://<orgname>-<account_name>.snowflakecomputing.com/api/v2'


paths:
  /statements:
    post:
      summary: Execute a SQL statement in Snowflake
      description: This endpoint allows users to execute a SQL statement in Snowflake, specifying the warehouse and roles to use.
      operationId: runQuery
      tags:
        - Statements
      requestBody:
        required: true
        content:
          application/json:
            schema:
              type: object
              properties:
                warehouse:
                  type: string
                  description: The name of the Snowflake warehouse to use for the statement execution.
                role:
                  type: string
                  description: The Snowflake role to assume for the statement execution.
                statement:
                  type: string
                  description: The SQL statement to execute.
              required:
                - warehouse
                - role
                - statement
      responses:
        '200':
          description: Successful execution of the SQL statement.
          content:
            application/json:
              schema:
                type: object
                properties:
                  status:
                    type: string
                  data:
                    type: object
                    additionalProperties: true
        '400':
          description: Bad request, e.g., invalid SQL statement or missing parameters.
        '401':
          description: Authentication error, invalid API credentials.
        '500':
          description: Internal server error.

## 2. 配置 Snowflake 集成

以下是与该第三方应用设置身份验证的说明。有疑问？请查看[入门示例](https://platform.openai.com/docs/actions/getting-started)，更详细地了解这一步是如何进行的。

### 为 ChatGPT 配置 IP 白名单
如果 Snowflake 账户设置了按 IP 限制连接的网络策略，可能需要为 ChatGPT 添加例外。
* 查阅 Snowflake 关于[网络策略](https://docs.snowflake.com/en/user-guide/network-policies)的文档
* 进入 Snowflake Worksheets
* 使用[此处](https://platform.openai.com/docs/actions/production/ip-egress-ranges#ip-egress-ranges)列出的 ChatGPT 出口 IP 范围创建网络规则
* 创建相应的网络策略

In [ ]:
## ChatGPT IP ranges available at https://openai.com/chatgpt-actions.json
CREATE NETWORK RULE chatgpt_network_rule
  MODE = INGRESS
  TYPE = IPV4
  VALUE_LIST = ('23.102.140.112/28',...,'40.84.180.128/28');

CREATE NETWORK POLICY chatgpt_network_policy
  ALLOWED_NETWORK_RULE_LIST = ('chatgpt_network_rule');

网络策略可以在账户、安全集成和用户级别应用。最具体的网络策略会覆盖较通用的网络策略。根据这些策略的应用方式，除了安全集成之外，你可能还需要修改个别用户的策略。如果遇到此问题，你可能会看到 Snowflake 的错误代码 390422 或通用的 "Invalid Client" 错误。

### 创建安全集成
* 查阅 Snowflake OAuth 概述：[https://docs.snowflake.com/en/user-guide/oauth-snowflake-overview](https://docs.snowflake.com/en/user-guide/oauth-snowflake-overview)
* 通过[安全集成](https://docs.snowflake.com/en/sql-reference/sql/create-security-integration-oauth-snowflake)创建新的 OAuth 凭据——每个 OAuth 应用/自定义 GPT 都需要一个新的安全集成，因为 Snowflake 的 Redirect URI 与安全集成是一一对应的

In [ ]:
CREATE SECURITY INTEGRATION CHATGPT_INTEGRATION
  TYPE = OAUTH
  ENABLED = TRUE
  OAUTH_CLIENT = CUSTOM
  OAUTH_CLIENT_TYPE = 'CONFIDENTIAL'
  OAUTH_REDIRECT_URI = 'https://oauth.pstmn.io/v1/callback' --- // this is a temporary value while testing your integration. You will replace this with the value your GPT provides
  OAUTH_ISSUE_REFRESH_TOKENS = TRUE
  OAUTH_REFRESH_TOKEN_VALIDITY = 7776000
  NETWORK_POLICY = chatgpt_network_policy; --- // this line should only be included if you followed step 1 above

<details>
  <summary>可选：自动化网络规则配置</summary>
  
  ChatGPT 目前使用的出口 IP 地址已超过 100 个。该列表会不定期更新，且不会另行通知。为了及时跟进，我们可以每天获取该列表并将其应用到网络规则中。

  ### 允许出站流量访问 OpenAI 的网络规则
  ```sql
  CREATE OR REPLACE NETWORK RULE chatgpt_actions_rule
  MODE = EGRESS       -- outbound
  TYPE = HOST_PORT
  VALUE_LIST = ('openai.com:443');
  ```
   ### 用于应用该规则的访问集成
  ```sql
  CREATE OR REPLACE EXTERNAL ACCESS INTEGRATION chatgpt_actions_integration
  ALLOWED_NETWORK_RULES = (chatgpt_actions_rule)
  ENABLED = TRUE;
  ```

  ### 获取 IP 范围的 UDF
  ```sql
  CREATE OR REPLACE FUNCTION getChatGPTActionsAddresses()
  RETURNS ARRAY  -- array<varchar>
  LANGUAGE PYTHON
  RUNTIME_VERSION = 3.10
  PACKAGES = ('requests')
  EXTERNAL_ACCESS_INTEGRATIONS = (chatgpt_actions_integration)
  HANDLER = 'get_ip_address_ranges'
AS
$$
import requests

def get_ip_address_ranges():
    resp = requests.get("https://openai.com/chatgpt-actions.json", timeout=10)
    resp.raise_for_status()
    data = [entry["ipv4Prefix"] for entry in resp.json().get("prefixes", []) if "ipv4Prefix" in entry]
    return data
$$;
  ```
  ### 更新网络规则的存储过程
  ```sql
  CREATE OR REPLACE PROCEDURE update_chatgpt_network_rule()
  RETURNS STRING
  LANGUAGE SQL
AS
$$
DECLARE
  ip_list STRING;
BEGIN
  -- Properly quote the IPs for use in VALUE_LIST
  ip_list := '''' || ARRAY_TO_STRING(getChatGPTActionsAddresses(), ''',''') || '''';

  -- Run the dynamic SQL to update the rule
  EXECUTE IMMEDIATE
    'ALTER NETWORK RULE chatgpt_network_rule SET VALUE_LIST = (' || ip_list || ')';

  RETURN 'chatgpt_network_rule updated with ' || ARRAY_SIZE(getChatGPTActionsAddresses()) || ' entries';
END;
$$;
  ```

  ### 调用存储过程
  ```sql
  CALL update_chatgpt_network_rule();
  ```

  ### 每天太平洋时间早上 6 点运行该存储过程
  ```sql
  CREATE OR REPLACE TASK auto_update_chatgpt_network_rule
  WAREHOUSE = COMPUTE_WH
  SCHEDULE = 'USING CRON 0 6 * * * America/Los_Angeles'
AS
  CALL update_chatgpt_network_rule();
  ```
</details>

## 3. 配置 GPT Action 身份验证
### 从 Snowflake 收集关键信息
* 获取你的 OAuth Client ID、Auth URL 和 Token URL


In [ ]:
DESCRIBE SECURITY INTEGRATION CHATGPT_INTEGRATION;


你会在以下 3 行中找到所需信息：


![../../../images/snowflake_direct_oauth.png](../../../images/snowflake_direct_oauth.png)


* 使用 SHOW_OAUTH_CLIENT_SECRETS 获取你的 OAuth Client Secret

In [ ]:
SELECT 
trim(parse_json(SYSTEM$SHOW_OAUTH_CLIENT_SECRETS('CHATGPT_INTEGRATION')):OAUTH_CLIENT_ID) AS OAUTH_CLIENT_ID
, trim(parse_json(SYSTEM$SHOW_OAUTH_CLIENT_SECRETS('CHATGPT_INTEGRATION')):OAUTH_CLIENT_SECRET) AS OAUTH_CLIENT_SECRET;

现在是[在 Postman 中测试 Snowflake 集成](https://community.snowflake.com/s/article/How-to-configure-postman-for-testing-SQL-API-with-OAuth)的好时机。如果你为安全集成配置了网络策略，请确保其中包含你用于测试的机器的 IP。

### 在 GPT Action 身份验证中设置 OAuth 值

在 ChatGPT 中，点击 "Authentication" 并选择 "OAuth"。输入以下信息。

| 表单字段 | 值  |
| -------- | -------- |
| Authentication Type   | OAuth   |
| Client ID   | 来自 SHOW_OAUTH_CLIENT_SECRETS 的 OAUTH_CLIENT_ID   |
| Client Secret   | 来自 SHOW_OAUTH_CLIENT_SECRETS 的 OAUTH_CLIENT_SECRET   |
| Authorization URL   | 来自 DESCRIBE SECURITY INTEGRATION 的 OAUTH_AUTHORIZATION_ENDPOINT |
| Token URL   | 来自 DESCRIBE SECURITY INTEGRATION 的 OAUTH_TOKEN_ENDPOINT   |
| Scope   | session:role:CHATGPT_INTEGRATION_ROLE*   |
| Token Exchange Method   | Default (POST Request)   |


*Snowflake 的 scope 以 `session:role:<your_role>` 格式传递角色，例如 `session:role:CHATGPT_INTEGRATION_ROLE`。你也可以将此字段留空，并在 GPT 指令中指定角色；但在这里添加后，角色会包含在 OAuth 授权同意请求中，有时会更可靠。 

### 4. 更新 Snowflake 集成的 Redirect URI


在 ChatGPT 中设置好身份验证后，在应用中按照以下步骤完成 Action 的配置。

* 从 GPT Action 中复制回调 URL
* 将安全集成中的 Redirect URI 更新为 ChatGPT 中提供的回调 URL。


In [ ]:
ALTER SECURITY INTEGRATION CHATGPT_INTEGRATION SET OAUTH_REDIRECT_URI='https://chat.openai.com/aip/<callback_id>/oauth/callback';

### 常见问题与故障排除

* 本指南旨在说明一般概念，仅供参考。我们无法为第三方 API 集成提供全面支持。 
* 如果你更新了 YAML，回调 url 可能会改变，修改时请仔细确认其是否正确。
* _回调 URL 错误：_ 如果在 ChatGPT 中遇到回调 URL 错误，请仔细查看上面的 Action 后续步骤。你需要将回调 URL 直接添加到安全集成中，Action 才能正确完成身份验证
* _Schema 调用了错误的仓库或数据库：_ 如果 ChatGPT 调用了错误的仓库或数据库，可以考虑更新指令，使其更加明确：(a) 应调用哪个仓库/数据库，或 (b) 要求用户在运行查询前提供这些确切信息



_有希望我们优先支持的集成吗？我们的集成中有错误吗？请在我们的 GitHub 上提交 PR 或 issue，我们会查看。_